# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [14]:
!git clone https://github.com/abdulmanan2728/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 336, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 336 (delta 141), reused 107 (delta 101), pack-reused 160 (from 1)
Receiving objects: 100% (336/336), 1.90 MiB | 12.88 MiB/s, done.
Resolving deltas: 100% (190/190), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [15]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)
df.head(3)

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. My lane as an ML task (type)

This is a **scoring** task built on top of binary classification. Per the framing skill's own mapping table, "Will this one decline / recover?" maps to classification with a target drawn from an OBSERVED outcome. The model itself solves a classification problem — will a page's impressions decline or not — but the deliverable is a ranked score that sorts pages into a weekly review queue; the action comes from where a page lands in that ranking, not from the raw yes/no label alone.

**Frame:** For an SEO analyst deciding which pages to review this week, we will build a ranked score from search performance data, predicting whether a content item is declining (using the dataset's observed trend label), measured by precision against the base rate on a client-grouped holdout. A wrong call costs wasted analyst time (false positive) or a missed decline that persists another cycle (false negative). A plain rule isn't enough because decline is driven by several tangled, shifting signals — not one clean threshold. We will claim only observed, directional, decision-support results.

**Ties to a real content action:** pages scoring highest get added to next week's manual review queue for an SEO analyst to check first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [16]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [17]:
# trend_direction is categorical — check its actual values before building a label from it
print(df['trend_direction'].unique())

['down' 'stable' 'new' 'up' 'flat']


In [18]:
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

label_check = df[['content_id', 'trend_pct', 'trend_direction', 'is_declining']].head(10)
label_check

,content_id,trend_pct,trend_direction,is_declining
0,content_304f48230142,-41.4,down,1
1,content_a1fb4e703a9e,-57.7,down,1
2,content_9aa793d4d895,-60.9,down,1
3,content_331d6c4de07b,-13.8,stable,0
4,content_d99b7a2d90ca,-34.7,down,1
5,content_d4084a4bc775,-38.9,down,1
6,content_9a34b442b552,-92.3,down,1
7,content_a63219c6e95a,0.6,stable,0
8,content_5e6c160719bc,-58.8,down,1
9,content_c27558df2b0c,-29.2,down,1


## 2. Target or proxy

The target is `is_declining`, built from the dataset's existing `trend_direction`
column (itself computed from `trend_pct`). The actual category values are `down`,
`stable`, `new`, `up`, and `flat` — I define `is_declining` as `trend_direction ==
'down'`. Per the data dictionary, `trend_direction` and `trend_pct` must NEVER be
used as features — only as the source of the label. This label is grounded in an
observed trailing-90-day performance trend, not an opinion or hand-labeled judgment.

One assumption worth naming: `new` content is treated as not-declining by default,
since it doesn't yet have enough history to show a real trend — this may deserve its
own category in later weeks rather than being lumped in with genuinely stable pages.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [19]:
base_rate = df['is_declining'].mean()
print(f"share of pages labeled declining: {base_rate:.3f}")

share of pages labeled declining: 0.542


## 3. Success metric

Precision on the "declining" class, measured against the base rate — matching the
metric the framing skill recommends for this task type (precision/recall vs. base
rate). The base rate here is 0.542 (54% of pages are labeled declining), so accuracy
alone would be a weak metric — a model that always guessed "declining" would already
score ~54% by doing nothing useful. Precision matches the actual cost tradeoff better:
a false positive wastes an analyst's time reviewing a page that was fine, which is the
mistake I'd rather avoid. Any split used later has to be client-grouped, not random —
a random split lets the same client leak across train/test and inflates the number.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [20]:
dupe_check = df['content_id'].duplicated().sum()
print(f"duplicate content_id rows: {dupe_check}")  # should be 0

unit_check = df[['content_id', 'client_id', 'ctr', 'avg_position', 'trend_pct', 'is_declining']].head(10)
unit_check

duplicate content_id rows: 0


,content_id,client_id,ctr,avg_position,trend_pct,is_declining
0,content_304f48230142,client_f369cb89fc,0.76,10.6,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,0.05,20.3,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.09,36.5,-60.9,1
3,content_331d6c4de07b,client_19581e27de,0.49,6.2,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.13,44.0,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,0.03,8.5,-38.9,1
6,content_9a34b442b552,client_8722616204,0.00,7.0,-92.3,1
7,content_a63219c6e95a,client_19581e27de,0.06,21.2,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,0.09,46.0,-58.8,1
9,content_c27558df2b0c,client_19581e27de,0.16,4.9,-29.2,1


## 4. The unit of analysis, as a real dataframe

One row = one pseudonymized content item (32 clients, trailing-90-day metrics).

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [21]:
naive_flagged = (df['ctr'] < df['ctr'].median()).mean()
strict_flagged = df['is_declining'].mean()
print(f"naive rule (below-median CTR) flags: {naive_flagged:.3f}")
print(f"actual declining-label rate: {strict_flagged:.3f}")

naive rule (below-median CTR) flags: 0.494
actual declining-label rate: 0.542


## 5. Why ML beats a fixed rule here

A single hand-written rule — e.g. "flag any page with below-median CTR" — is too
blunt: decline isn't driven by one signal alone, it's a tangled combination of
momentum, click behavior, and ranking stability, each shifting differently across
content types and clients. The right weighting of those signals isn't obvious from
eyeballing thresholds, but a model can learn it from data and re-rank consistently —
which is exactly what a baseline rule vs. a trained model comparison is designed to
test in a later week.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.